# Pipeline ETL com IA Generativa 🏦

Projeto desenvolvido como parte do portfólio pessoal.

O objetivo é construir um pipeline ETL completo em Python que:
- **Extrai** dados de usuários de uma API pública
- **Transforma** os dados enriquecendo com perfil bancário gerado por IA
- **Carrega** os dados consolidados em múltiplos formatos para uso posterior

### Tecnologias utilizadas
- **Python** — linguagem principal
- **Groq API + LLaMA 3.3** — IA Generativa para enriquecimento e marketing bancário
- **Pandas** — manipulação e estruturação de dados
- **SQLite** — persistência relacional normalizada
- **JSONPlaceholder** — API pública de dados fictícios

---

### 1. Bibliotecas
Importação de todas as dependências necessárias para o projeto.

In [2]:
import requests as req
import pandas as pd
import sqlite3 as sql
import json
import time
import os
import re

from groq import Groq
from dotenv import load_dotenv

### 2. Configuração da API
Conexão segura com a API do Groq utilizando variável de ambiente via `.env`.  
A chave nunca fica exposta diretamente no código.

In [3]:
# Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

# Cria o cliente Groq com a chave do .env
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

### 3. Extract — Extração dos Dados

Primeira etapa do pipeline ETL.

Os IDs dos usuários são lidos a partir de um arquivo CSV e utilizados para buscar
os dados de cada usuário na API do JSONPlaceholder. Usuários não encontrados são
automaticamente filtrados da lista.

In [4]:
df_ID = pd.read_csv('users_id.csv')
user_ids = df_ID['userID'].to_list()

JSONPlaceholder = 'https://jsonplaceholder.typicode.com'

In [5]:
def get_user(id):
    response = req.get(f'{JSONPlaceholder}/users/{id}')
    return response.json() if response.status_code == 200 else None

# Walrus operator (:=) filtra automaticamente usuários não encontrados (None)
users = [user for id in user_ids if (user := get_user(id)) is not None] 

### 4. Transform — Enriquecimento e Marketing

Segunda etapa do pipeline ETL — dividida em duas funções:

- **`enrich_user_data`** → envia o perfil do usuário para a IA e recebe de volta
dados bancários fictícios e realistas como renda, score de crédito e perfil bancário

- **`generate_ai_msg`** → com o perfil já enriquecido, gera uma mensagem de
marketing bancário personalizada para cada cliente

In [6]:
def enrich_user_data(users):
    completion = client.chat.completions.create( 
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "Você é um especialista em análise de perfil bancário e geração de dados financeiros fictícios, mas realistas."
            },
            {
                "role": "user",
                "content": f"""
                            Com base neste perfil de usuário:
                            Nome: {users['name']}
                            Empresa: {users['company']['name']}
                            Website: {users['website']}

                            Gere dados bancários fictícios e realistas.
                            Responda APENAS com um JSON válido, sem texto adicional, contendo:
                            - monthly_income (float, use ponto como separador decimal. Ex: 8500.00)
                            - credit_score (int between 0 and 1000)
                            - banking_profile (string: "Conservador", "Moderado" ou "Investidor")
                            - has_debts (boolean: true ou false, em minúsculo)
                            - banking_products (list of strings)
                                                         
                            PERFIL BANCÁRIO:
                            - Varie os perfis de forma realista.
                            - É OBRIGATÓRIO variar entre os três perfis: "Conservador", "Moderado" e "Investidor".
                            - Analise a empresa e website para inferir o perfil correto.
                            
                            PRODUTOS BANCÁRIOS:
                            - Inclua apenas os nomes padronizados: "Conta Corrente", "Cartão de Crédito", 
                            "Previdência Privada", "Empréstimo Pessoal", "Investimentos", "Ações", 
                            "Fundos Imobiliários".
                            - Varie a quantidade de produtos de acordo com o perfil bancário do usuário.
                            - Não crie nomenclaturas diferentes das listadas acima.
                            - Siga esta lógica de coerência:
                                * Conservador: produtos simples como Conta Corrente, Poupança, Cartão de Crédito
                                * Moderado:    produtos intermediários como Investimentos, Previdência Privada
                                * Investidor:  produtos avançados como Ações, Fundos Imobiliários
                            
                            RENDA E SCORE:
                            - Diversifique bastante a renda mensal e o score de crédito.
                            - Evite que todos os usuários tenham valores semelhantes. 
                            - A renda mensal, assim como o score de crédito, deve variar de acordo com o perfil do usuário e a empresa onde trabalhar.
                                * Conservador: renda entre R$ 1.500 e R$ 4.000, score entre 300 e 600
                                * Moderado:    renda entre R$ 4.000 e R$ 10.000, score entre 600 e 800
                                * Investidor:  renda entre R$ 10.000 e R$ 50.000, score entre 800 e 1000
                            
                """
            }
        ]
    )
    # re.sub necessário — modelo ocasionalmente retorna JSON envolto em markdown
    content = re.sub(r"```json|```", "", completion.choices[0].message.content).strip()
    enriched_data = json.loads(content)
    return {**users, **enriched_data}


def generate_ai_msg(users):  
    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "Você é um especialista em marketing bancário."
            },
            {
                "role": "user",
                "content": f"""
                    Crie uma mensagem sobre a importância dos investimentos para {users['name']} 
                    da empresa {users['company']['name']} com base no perfil bancário: {users['banking_profile']}.
                    
                    REGRAS:
                    - Máximo de 200 caracteres.
                    - Responda APENAS com a mensagem, sem aspas, sem quebras de linha.
                    - A mensagem deve ser personalizada e relevante para o perfil do usuário.
                """
                
            }
        ]
    )
    
    # strip e replace garantem resposta limpa independente do comportamento do modelo
    ai_msg = (completion.choices[0].message.content
              .strip()
              .replace('"', '')
              .replace('\n', ' '))
    
    return {**users, **{"ai_msg": ai_msg}}

#### 4.1 Execução do Pipeline

Processa cada usuário em sequência — enriquece os dados e gera o marketing bancário.
Um delay de 2 segundos entre cada requisição é aplicado para respeitar o Rate Limit da API.

In [7]:
final_users = []

for user in users:
    enriched_user = enrich_user_data(user) 
    ai_msg = generate_ai_msg(enriched_user) 
    final_users.append(ai_msg)
  
    print(f"Usuário: {ai_msg['name']}")
    print(f"Perfil: {ai_msg['banking_profile']}")
    print(f"Mensagem: {ai_msg['ai_msg']}")
    print("----")

    time.sleep(2) # Evita rate limit da API do Groq

Usuário: Leanne Graham
Perfil: Investidor
Mensagem: Leanne Graham, maximize seu patrimônio com nossos investimentos personalizados, garantindo rentabilidade e segurança para o futuro da Romaguera-Crona.
----
Usuário: Ervin Howell
Perfil: Investidor
Mensagem: Ervin, diversifique sua carteira com nossos investimentos de alto rendimento e proteja seu patrimônio.
----
Usuário: Clementine Bauch
Perfil: Moderado
Mensagem: Clementine, com uma abordagem moderada, seus investimentos podem trazer estabilidade e crescimento a longo prazo.
----
Usuário: Patricia Lebsack
Perfil: Investidor
Mensagem: Patricia, aproveite ao máximo seus investimentos com rentabilidade alta e segurança.
----
Usuário: Chelsey Dietrich
Perfil: Investidor
Mensagem: Chelsey, como investidor, você sabe que diversificar sua carteira é chave para o sucesso financeiro, vamos conversar sobre novas oportunidades de investimento para Keebler LLC.
----
Usuário: Mrs. Dennis Schulist
Perfil: Investidor
Mensagem: Diversifique sua car

### 5. Load — Persistência dos Dados
Terceira e última etapa do pipeline ETL.

Os dados consolidados são persistidos em múltiplos formatos, cada um pensado
para um tipo diferente de consumidor:

- **JSON** — para consumo por APIs e sistemas externos
- **CSV** — para análise em ferramentas como Excel e Power BI
- **SQLite** — banco relacional normalizado em 3 tabelas: `users`, `address` e `products`

#### 5.1 JSON

In [17]:
# Persiste os dados consolidados em JSON
with open('banking_users.json', 'w', encoding='utf-8') as f:
    json.dump(final_users, f, indent=2, ensure_ascii=False)

print(f"✅ JSON Salvo: {len(final_users)} registros")  
final_users[0]


✅ JSON Salvo: 10 registros


{'id': 1,
 'name': 'Leanne Graham',
 'username': 'Bret',
 'email': 'Sincere@april.biz',
 'address': {'street': 'Kulas Light',
  'suite': 'Apt. 556',
  'city': 'Gwenborough',
  'zipcode': '92998-3874',
  'geo': {'lat': '-37.3159', 'lng': '81.1496'}},
 'phone': '1-770-736-8031 x56442',
 'website': 'hildegard.org',
 'company': {'name': 'Romaguera-Crona',
  'catchPhrase': 'Multi-layered client-server neural-net',
  'bs': 'harness real-time e-markets'},
 'monthly_income': 8200.0,
 'credit_score': 920,
 'banking_profile': 'Investidor',
 'has_debts': False,
 'banking_products': ['Conta Corrente',
  'Cartão de Crédito',
  'Investimentos',
  'Ações',
  'Fundos Imobiliários',
  'Previdência Privada'],
 'ai_msg': 'Leanne Graham, maximize seu patrimônio com nossos investimentos personalizados, garantindo rentabilidade e segurança para o futuro da Romaguera-Crona.'}

#### 5.2 CSV
Dados estruturados para análise. Colunas aninhadas (address, company, banking_products)
são expandidas e os produtos bancários transformados em one-hot encoding para facilitar
filtragem no Excel.

In [18]:
banking_users = pd.DataFrame(final_users)
address = pd.DataFrame([user['address'] for user in final_users])
banking_users['company_name'] = [user['company']['name'] for user in final_users]

# One-Hot Encoding nos produtos para facilitar filtragem no Excel
banking_products = banking_users['banking_products'].str.join('|').str.get_dummies().replace({0: 'Não', 1: 'Sim'})

banking_users = (pd.concat([banking_users, address, banking_products], axis=1)
                .drop(columns=['address', 'company', 'banking_products', 'geo'])
                .rename(columns={'id': 'user_id', 'Ações': 'stocks', 'Cartão de Crédito': 'credit_card', 'Conta Corrente': 'checking_account', 'Empréstimo Pessoal': 'personal_loan', 'Fundos Imobiliários': 'real_estate_funds', 'Investimentos': 'investments', 'Previdência Privada': 'private_pension'}))

# utf-8-sig necessário para Excel brasileiro reconhecer acentuação corretamente
banking_users.to_csv('banking_users.csv', index=False, encoding="utf-8-sig", sep=';')
print(f"✅ CSV salvo — {len(banking_users)} registros exportados")

banking_users.head()

✅ CSV salvo — 10 registros exportados


,user_id,name,username,email,phone,website,monthly_income,credit_score,banking_profile,has_debts,...,street,suite,city,zipcode,stocks,credit_card,checking_account,real_estate_funds,investments,private_pension
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,8200.0,920,Investidor,False,...,Kulas Light,Apt. 556,Gwenborough,92998-3874,Sim,Sim,Sim,Sim,Sim,Sim
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,12000.0,920,Investidor,False,...,Victor Plains,Suite 879,Wisokyburgh,90566-7771,Sim,Sim,Sim,Sim,Sim,Sim
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,6500.0,720,Moderado,False,...,Douglas Extension,Suite 847,McKenziehaven,59590-4157,Não,Sim,Sim,Não,Sim,Sim
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,12000.0,920,Investidor,False,...,Hoeger Mall,Apt. 692,South Elvis,53919-4257,Sim,Sim,Não,Sim,Sim,Sim
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,12000.0,920,Investidor,False,...,Skiles Walks,Suite 351,Roscoeview,33263,Sim,Sim,Sim,Sim,Sim,Sim


#### 5.3 SQLite
Dados normalizados em 3 tabelas relacionadas pelo `user_id`, seguindo boas práticas
de modelagem relacional.

In [19]:
df_users = banking_users[['user_id', 'name', 'username', 'email', 'phone', 'website', 
                          'company_name', 'monthly_income', 'credit_score',
                          'banking_profile', 'has_debts', 'ai_msg']]

df_address = banking_users[['user_id','street', 'suite', 'city', 'zipcode']]

# Normaliza produtos - uma linha por produto para evitar arrays no banco
df_products = pd.DataFrame([
    {'user_id': user['id'], 'products': product} 
    for user in final_users 
    for product in user['banking_products']
])

conn = sql.connect("banking_users.db")

df_users.to_sql("users", conn, if_exists="replace", index=False)
df_address.to_sql("address", conn, if_exists="replace", index=False)
df_products.to_sql("products", conn, if_exists="replace", index=False)

check_users = pd.read_sql("SELECT * FROM users", conn)
check_address = pd.read_sql("SELECT * FROM address", conn)
check_products = pd.read_sql("SELECT * FROM products", conn)

conn.close()

print(f"✅ Users:    {len(check_users)} registros")
print(f"✅ Address:  {len(check_address)} registros")
print(f"✅ Products: {len(check_products)} registros")

check_users.head()

✅ Users:    10 registros
✅ Address:  10 registros
✅ Products: 49 registros


,user_id,name,username,email,phone,website,company_name,monthly_income,credit_score,banking_profile,has_debts,ai_msg
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Romaguera-Crona,8200.0,920,Investidor,0,"Leanne Graham, maximize seu patrimônio com nos..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Deckow-Crist,12000.0,920,Investidor,0,"Ervin, diversifique sua carteira com nossos in..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Romaguera-Jacobson,6500.0,720,Moderado,0,"Clementine, com uma abordagem moderada, seus i..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Robel-Corkery,12000.0,920,Investidor,0,"Patricia, aproveite ao máximo seus investiment..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Keebler LLC,12000.0,920,Investidor,0,"Chelsey, como investidor, você sabe que divers..."
